# **Project 3: Baseline RAG (no agent)**

The simplest version: **retrieve top-5 sections -> ask the LLM to answer from them.** No grading, no rewriting, no checking.

We build it first because the agent in Project 4 has to **beat** it, and we need the same prompt and the same model for a fair comparison.

In [1]:
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
import os

load_dotenv()

if os.environ['OPENROUTER_API_KEY']:
    print("OpenRouter API Key is set")
else:
    raise ValueError("OpenRouter API Key is not set")

OpenRouter API Key is set


In [2]:
llm = ChatOpenRouter(model='gpt-5-nano')
llm.invoke("Say hello in one short sentence").content

'Hello there.'

### **Load the vector store from Project 2**

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5", encode_kwargs={"normalize_embeddings": True})

vectorstore = Chroma(
    collection_name="employment_act",
    embedding_function=embeddings,
    persist_directory="../chroma_db",
)

print(vectorstore._collection.count(), "chunks in the store")

c:\Users\User\employ-law-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3129.77it/s]


168 chunks in the store


### **Helper functions**
These same helpers are used again in Project 4, so both systems read the same context in the same format.

In [4]:
TOP_K = 5
INSUFFICIENT = "INSUFFICIENT_EVIDENCE"

def retrieve_docs(query, k=TOP_K):
    # search the vector store and keep only what we need from each result
    results = vectorstore.similarity_search(query, k=k)
    return [{"content": r.page_content,
             "section_no": r.metadata["section_no"],
             "section_title": r.metadata["section_title"]} for r in results]

def format_context(docs):
    # one block per section, labelled so the LLM can cite it, e.g. [s.60A]
    return "\n\n".join(f"[s.{d['section_no']}] {d['section_title']}\n{d['content']}" for d in docs)

def get_citations(answer):
    # find every [s.xxx] label in the answer (plain string methods, no regex)
    found = []
    for part in answer.split("[")[1:]:
        tag = part.split("]")[0]
        if tag.startswith("s."):
            found.append(tag)
    return sorted(set(found))

### **The answer chain (prompt | llm)**
Same pattern as notebook 4 (prompts and chains).

In [5]:
from langchain_core.prompts import ChatPromptTemplate

generate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You explain the Malaysian Employment Act 1955. Answer ONLY using the context. "
               "Cite every claim with its label in square brackets, exactly as given, for example [s.60A]. "
               "Do not cite anything that is not in the context. "
               "If the context does not contain enough information, reply exactly: INSUFFICIENT_EVIDENCE. "
               "This is general information, not legal advice."),
    ("human", "Context:\n{context}\n\nQuestion: {question}\n{feedback}"),
])

generate_chain = generate_prompt | llm

In [6]:
import time

def baseline_answer(question):
    start = time.time()

    docs = retrieve_docs(question)
    answer = generate_chain.invoke({"context": format_context(docs), "question": question, "feedback": ""}).content.strip()

    return {"answer": answer,
            "documents": docs,
            "citations": get_citations(answer),
            "abstained": INSUFFICIENT in answer,
            "latency": time.time() - start}

### **Try it on a few questions**
For each one check: was the right section retrieved, and was it cited?

In [7]:
def try_baseline(question):
    r = baseline_answer(question)
    print("Q:", question)
    print("retrieved:", [f"s.{d['section_no']}" for d in r["documents"]])
    print("cited:", r["citations"], "| abstained:", r["abstained"], f"| {r['latency']:.1f}s")
    print(r["answer"])
    print("-" * 70)

try_baseline("What is the maximum normal working week?")
try_baseline("How much notice must be given to terminate a contract of service if the contract says nothing?")
try_baseline("What are my rights as a worker in Sabah under this Act?")
try_baseline("What is the latest 2027 income tax relief limit?")

Q: What is the maximum normal working week?
retrieved: ['s.60', 's.60A', 's.60C', 's.60D', 's.60A']
cited: ['s.60A'] | abstained: False | 12.0s
Forty-five hours per week. [s.60A]
----------------------------------------------------------------------
Q: How much notice must be given to terminate a contract of service if the contract says nothing?
retrieved: ['s.13', 's.12', 's.11', 's.57', 's.21']
cited: ['s.12'] | abstained: False | 23.6s
- If the contract says nothing and the employee has been employed for less than two years: four weeks’ notice. [s.12]
- If the employee has been employed for two years or more but less than five years: six weeks’ notice. [s.12]
- If the employee has been employed for five years or more: eight weeks’ notice. [s.12]
----------------------------------------------------------------------
Q: What are my rights as a worker in Sabah under this Act?
retrieved: ['s.1', 's.60A', 's.2', 's.101C', 's.103']
cited: ['s.1', 's.101C', 's.60A'] | abstained: False | 55

### **What you should notice**
The baseline answers *everything*, including the Sabah and tax questions, because nothing stops it.
That is exactly the weakness the agent in Project 4 is built to fix. Write down any wrong or unsupported answers you find in `notes.md`.